## PDP and Joint-PDP via glex — IEEE-CIS & KDD Cup

Reproduces the PDP/J-PDP experiments from `BACKUP_PDP_woodelf_vs_sota.ipynb`
using the **glex** functional decomposition instead of woodelf.

Models match the original: XGBoost with `max_depth=6`, `nrounds=100`.

**Experiments (both datasets):**
- PDP k=5 / k=10 / k=100 — equally-spaced grid between 5th–95th percentile
- Full PDP — grid = all unique split thresholds from the fitted trees
- Joint PDP k=5 — all feature pairs, 25 grid combinations each

**Mount Google Drive before running** (Colab sidebar → Files → Mount Drive).

In [1]:
install.packages(c('xgboost', 'arrow', 'data.table', 'remotes'),
                 repos = 'https://cloud.r-project.org', quiet = TRUE)
remotes::install_github('PlantedML/glex', quiet = TRUE)

also installing the dependency ‘assertthat’


Installing 13 packages: rlang, glue, cli, vctrs, cpp11, S7, iterators, backports, Rcpp, scico, ggplot2, foreach, checkmate



In [2]:
library(xgboost)
library(glex)
library(arrow)
library(data.table)


Attaching package: ‘arrow’


The following object is masked from ‘package:utils’:

    timestamp




## `glex_xy`: glex with separate background data

Calls `glex:::tree_fun_emp_fastPD` directly so that `x_explain` (the PDP grid)
and `x_background` (the marginal reference distribution) can be different matrices.

In [3]:
glex_xy <- function(model, x_explain, x_background, max_interaction = NULL) {
  if (!is.matrix(x_explain))    x_explain    <- as.matrix(x_explain)
  if (!is.matrix(x_background)) x_background <- as.matrix(x_background)
  if (is.null(max_interaction))  max_interaction <- 9999

  trees <- xgboost::xgb.model.dt.tree(model = model, use_int_id = TRUE)
  trees$Type <- '<'

  unique_feats <- unique(trees$Feature)
  unique_feats <- unique_feats[unique_feats != 'Leaf']
  if (suppressWarnings(all(!is.na(as.integer(unique_feats))))) {
    trees[Feature == 'Leaf', Feature_num := 0L]
    trees[Feature != 'Leaf', Feature_num := as.integer(Feature) + 1L]
  } else {
    trees[, Feature_num := as.integer(
      factor(Feature, levels = c('Leaf', colnames(x_explain)))) - 1L]
  }

  all_S <- unique(do.call(c, lapply(0:max(trees$Tree), function(t) {
    feats <- trees[Tree == t & Feature_num > 0, sort(unique(Feature_num))]
    glex:::get_all_subsets_cpp(feats, max_interaction)
  })))

  m_all     <- matrix(0, nrow = nrow(x_explain), ncol = length(all_S))
  col_names <- NULL
  for (t in 0:max(trees$Tree)) {
    m_tree <- glex:::tree_fun_emp_fastPD(
      t, trees, x_explain, x_background, all_S, max_interaction
    )
    if (is.null(col_names)) col_names <- colnames(m_tree)
    m_all <- m_all + m_tree
  }
  colnames(m_all) <- col_names

  base_score <- glex:::get_xgb_base_score(model)
  list(
    m         = data.table::setDT(as.data.frame(m_all[, -1])),
    intercept = unique(m_all[, 1]) + base_score,
    x         = data.table::setDT(as.data.frame(x_explain))
  )
}

## Helper functions

In [4]:
# ── Grid builders ─────────────────────────────────────────────────────────────

# k equally-spaced points between the p-th and (1-p)-th percentile of each feature
build_pdp_grid <- function(data, k, percentiles = c(0.05, 0.95)) {
  data <- as.matrix(data)
  grid <- matrix(NA_real_, nrow = k, ncol = ncol(data))
  colnames(grid) <- colnames(data)
  for (j in seq_len(ncol(data))) {
    col <- data[, j]
    col <- col[!is.na(col)]
    lo  <- quantile(col, percentiles[1], names = FALSE)
    hi  <- quantile(col, percentiles[2], names = FALSE)
    grid[, j] <- seq(lo, hi, length.out = k)
  }
  grid
}

# All unique split thresholds per feature extracted from the fitted XGBoost model
build_full_pdp_grid <- function(model, feature_names) {
  trees    <- xgb.model.dt.tree(model = model)
  internal <- trees[Feature != 'Leaf']

  th <- lapply(feature_names, function(f) sort(unique(internal[Feature == f, Split])))
  names(th) <- feature_names

  max_len <- max(sapply(th, length), 1L)
  grid <- matrix(0, nrow = max_len, ncol = length(feature_names))
  colnames(grid) <- feature_names
  for (f in feature_names) {
    v <- th[[f]];  n <- length(v)
    if (n > 0) {
      grid[seq_len(n), f] <- v
      if (n < max_len) grid[(n + 1):max_len, f] <- v[n]  # pad with last threshold
    }
  }
  grid
}

# ── Joint-PDP grid (bit construction) ─────────────────────────────────────────
# Mirrors woodelf's build_points_for_joint_pdp.
# For D = ceil(log2(F)) and k points per feature, creates D*k^2 rows.
# For any pair (fi, fj), rows (h*k^2+1):(h+1)*k^2 form the k x k Cartesian grid,
# where h = first bit position (MSB first) at which binary(fi) and binary(fj) differ.

bits_msb_first <- function(n, D) {
  bs <- integer(D)
  for (i in D:1) { bs[i] <- n %% 2; n <- n %/% 2 }
  rev(bs)
}

first_different_bit <- function(i, j, D) {
  # 0-indexed i, j; returns 0-indexed bit position (MSB = 0)
  b1 <- bits_msb_first(i, D);  b2 <- bits_msb_first(j, D)
  for (pos in seq_len(D)) if (b1[pos] != b2[pos]) return(pos - 1L)
  stop('i and j must differ')
}

build_joint_pdp_grid <- function(base_grid) {
  k <- nrow(base_grid);  F <- ncol(base_grid)
  D <- ceiling(log2(max(F, 2L)))
  result <- matrix(NA_real_, nrow = D * k^2, ncol = F)
  colnames(result) <- colnames(base_grid)
  for (i in seq_len(F)) {
    bs       <- bits_msb_first(i - 1L, D)  # 0-indexed feature
    col_vals <- numeric(0)
    for (b in bs) {
      if (b == 0L) col_vals <- c(col_vals, rep(base_grid[, i], times = k))   # tile
      else         col_vals <- c(col_vals, rep(base_grid[, i], each  = k))   # repeat
    }
    result[, i] <- col_vals
  }
  result
}

jpdp_rows <- function(i0, j0, D, k) {
  # i0, j0: 0-indexed feature numbers
  # Returns 1-indexed row range for this pair in the joint grid
  h     <- first_different_bit(i0, j0, D)
  start <- h * k^2 + 1L
  seq(start, start + k^2 - 1L)
}

# ── Timing wrapper ─────────────────────────────────────────────────────────────
time_glex_xy <- function(model, x_explain, x_background, label, max_interaction = 1L) {
  cat(sprintf('  %-20s n_exp=%d  n_bg=%d  max_s=%d ... ',
              label, nrow(x_explain), nrow(x_background), max_interaction))
  t0 <- proc.time()[['elapsed']]
  gl <- glex_xy(model, x_explain, x_background, max_interaction = max_interaction)
  dt <- proc.time()[['elapsed']] - t0
  cat(sprintf('%.2f sec\n', dt))
  list(result = gl, time = dt)
}

---
## Dataset 1: IEEE-CIS Fraud Detection

In [9]:
# create this folder locally, after starting the colab session and drag their the files.
# Sorry for the trouble, I couldn't make it mount to the drive in R.
DATA_DIR <- '/content/PDP_Data'

ieee_train_raw <- as.data.frame(arrow::read_parquet(
  file.path(DATA_DIR, 'ieee_cis_fraud_train.parquet')))

# Separate target; sanitise feature names for XGBoost
ieee_y      <- ieee_train_raw[['isFraud']]
ieee_X_raw  <- ieee_train_raw[, setdiff(colnames(ieee_train_raw), 'isFraud')]
colnames(ieee_X_raw) <- make.names(colnames(ieee_X_raw))
ieee_X      <- as.matrix(ieee_X_raw)
ieee_feats  <- colnames(ieee_X)
ieee_X_bg   <- ieee_X  # use full training set as background

cat(sprintf('IEEE-CIS: %d rows x %d features\n', nrow(ieee_X), ncol(ieee_X)))

IEEE-CIS: 472432 rows x 398 features


In [6]:
cat('Training IEEE-CIS model...\n')
t0 <- proc.time()[['elapsed']]
ieee_model <- xgboost(
  x         = ieee_X,
  y         = ieee_y,
  max_depth = 6,
  eta       = 0.1,
  objective = 'reg:squarederror',
  nrounds   = 100,
  verbose   = 0
)
cat(sprintf('Training done in %.1f sec\n', proc.time()[['elapsed']] - t0))

Training IEEE-CIS model...


Warning message in throw_err_or_depr_msg("Passed unrecognized parameters: ", paste(head(names_unrecognized), :
“Passed unrecognized parameters: verbose. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'eta' has been renamed to 'learning_rate'. This warning will become an error in a future version.”


Training done in 20.0 sec


In [7]:
cat('\n=== IEEE-CIS: building grids ===\n')
ieee_grid_5   <- build_pdp_grid(ieee_X, k = 5)
ieee_grid_10  <- build_pdp_grid(ieee_X, k = 10)
ieee_grid_100 <- build_pdp_grid(ieee_X, k = 100)
ieee_grid_full <- build_full_pdp_grid(ieee_model, ieee_feats)
ieee_grid_joint5 <- build_joint_pdp_grid(ieee_grid_5)  # D*25 rows

cat(sprintf('  k=5 grid:     %d x %d\n', nrow(ieee_grid_5),   ncol(ieee_grid_5)))
cat(sprintf('  k=10 grid:    %d x %d\n', nrow(ieee_grid_10),  ncol(ieee_grid_10)))
cat(sprintf('  k=100 grid:   %d x %d\n', nrow(ieee_grid_100), ncol(ieee_grid_100)))
cat(sprintf('  full grid:    %d x %d (max thresholds per feature)\n',
            nrow(ieee_grid_full), ncol(ieee_grid_full)))
cat(sprintf('  joint k=5:    %d x %d\n', nrow(ieee_grid_joint5), ncol(ieee_grid_joint5)))


=== IEEE-CIS: building grids ===
  k=5 grid:     5 x 398
  k=10 grid:    10 x 398
  k=100 grid:   100 x 398
  full grid:    95 x 398 (max thresholds per feature)
  joint k=5:    225 x 398


In [10]:
cat('\n=== IEEE-CIS: glex timing ===\n')

ieee_res_pdp5   <- time_glex_xy(ieee_model, ieee_grid_5,      ieee_X_bg, 'PDP k=5',       max_interaction = 1L)
ieee_res_pdp10  <- time_glex_xy(ieee_model, ieee_grid_10,     ieee_X_bg, 'PDP k=10',      max_interaction = 1L)
ieee_res_pdp100 <- time_glex_xy(ieee_model, ieee_grid_100,    ieee_X_bg, 'PDP k=100',     max_interaction = 1L)
ieee_res_full   <- time_glex_xy(ieee_model, ieee_grid_full,   ieee_X_bg, 'Full PDP',      max_interaction = 1L)
ieee_res_jpdp5  <- time_glex_xy(ieee_model, ieee_grid_joint5, ieee_X_bg, 'Joint PDP k=5', max_interaction = 2L)

ieee_times <- c(
  'PDP k=5'       = ieee_res_pdp5$time,
  'PDP k=10'      = ieee_res_pdp10$time,
  'PDP k=100'     = ieee_res_pdp100$time,
  'Full PDP'      = ieee_res_full$time,
  'Joint PDP k=5' = ieee_res_jpdp5$time
)
cat('\n--- IEEE-CIS summary ---\n')
for (nm in names(ieee_times))
  cat(sprintf('  %-20s %.2f sec\n', nm, ieee_times[[nm]]))


=== IEEE-CIS: glex timing ===
  PDP k=5              n_exp=5  n_bg=472432  max_s=1 ... 98.56 sec
  PDP k=10             n_exp=10  n_bg=472432  max_s=1 ... 98.82 sec
  PDP k=100            n_exp=100  n_bg=472432  max_s=1 ... 99.09 sec
  Full PDP             n_exp=95  n_bg=472432  max_s=1 ... 98.41 sec
  Joint PDP k=5        n_exp=225  n_bg=472432  max_s=2 ... 129.83 sec

--- IEEE-CIS summary ---
  PDP k=5              98.56 sec
  PDP k=10             98.82 sec
  PDP k=100            99.09 sec
  Full PDP             98.41 sec
  Joint PDP k=5        129.83 sec


---
## Dataset 2: KDD Cup (Network Intrusion Detection)

In [12]:
kdd_train_raw <- as.data.frame(arrow::read_parquet(
  file.path(DATA_DIR, 'KDD_cup_train.parquet')))

kdd_y     <- kdd_train_raw[['target']]
kdd_X_raw <- kdd_train_raw[, setdiff(colnames(kdd_train_raw), 'target')]
colnames(kdd_X_raw) <- make.names(colnames(kdd_X_raw))
kdd_X     <- as.matrix(kdd_X_raw)
kdd_feats <- colnames(kdd_X)
kdd_X_bg  <- kdd_X  # use full training set as background

cat(sprintf('KDD Cup: %d rows x %d features\n', nrow(kdd_X), ncol(kdd_X)))

KDD Cup: 4898431 rows x 121 features


In [13]:
cat('Training KDD Cup model...\n')
t0 <- proc.time()[['elapsed']]
kdd_model <- xgboost(
  x         = kdd_X,
  y         = kdd_y,
  max_depth = 6,
  eta       = 0.1,
  objective = 'reg:squarederror',
  nrounds   = 100,
  verbose   = 0
)
cat(sprintf('Training done in %.1f sec\n', proc.time()[['elapsed']] - t0))

Training KDD Cup model...


Warning message in throw_err_or_depr_msg("Passed unrecognized parameters: ", paste(head(names_unrecognized), :
“Passed unrecognized parameters: verbose. This warning will become an error in a future version.”
Warning message in throw_err_or_depr_msg("Parameter '", match_old, "' has been renamed to '", :
“Parameter 'eta' has been renamed to 'learning_rate'. This warning will become an error in a future version.”


Training done in 43.6 sec


In [14]:
cat('\n=== KDD Cup: building grids ===\n')
kdd_grid_5    <- build_pdp_grid(kdd_X, k = 5)
kdd_grid_10   <- build_pdp_grid(kdd_X, k = 10)
kdd_grid_100  <- build_pdp_grid(kdd_X, k = 100)
kdd_grid_full <- build_full_pdp_grid(kdd_model, kdd_feats)
kdd_grid_joint5 <- build_joint_pdp_grid(kdd_grid_5)

cat(sprintf('  k=5 grid:     %d x %d\n', nrow(kdd_grid_5),   ncol(kdd_grid_5)))
cat(sprintf('  k=10 grid:    %d x %d\n', nrow(kdd_grid_10),  ncol(kdd_grid_10)))
cat(sprintf('  k=100 grid:   %d x %d\n', nrow(kdd_grid_100), ncol(kdd_grid_100)))
cat(sprintf('  full grid:    %d x %d\n', nrow(kdd_grid_full), ncol(kdd_grid_full)))
cat(sprintf('  joint k=5:    %d x %d\n', nrow(kdd_grid_joint5), ncol(kdd_grid_joint5)))


=== KDD Cup: building grids ===
  k=5 grid:     5 x 121
  k=10 grid:    10 x 121
  k=100 grid:   100 x 121
  full grid:    42 x 121
  joint k=5:    175 x 121


In [15]:
cat('\n=== KDD Cup: glex timing ===\n')

kdd_res_pdp5   <- time_glex_xy(kdd_model, kdd_grid_5,      kdd_X_bg, 'PDP k=5',       max_interaction = 1L)
kdd_res_pdp10  <- time_glex_xy(kdd_model, kdd_grid_10,     kdd_X_bg, 'PDP k=10',      max_interaction = 1L)
kdd_res_pdp100 <- time_glex_xy(kdd_model, kdd_grid_100,    kdd_X_bg, 'PDP k=100',     max_interaction = 1L)
kdd_res_full   <- time_glex_xy(kdd_model, kdd_grid_full,   kdd_X_bg, 'Full PDP',      max_interaction = 1L)
kdd_res_jpdp5  <- time_glex_xy(kdd_model, kdd_grid_joint5, kdd_X_bg, 'Joint PDP k=5', max_interaction = 2L)

kdd_times <- c(
  'PDP k=5'       = kdd_res_pdp5$time,
  'PDP k=10'      = kdd_res_pdp10$time,
  'PDP k=100'     = kdd_res_pdp100$time,
  'Full PDP'      = kdd_res_full$time,
  'Joint PDP k=5' = kdd_res_jpdp5$time
)
cat('\n--- KDD Cup summary ---\n')
for (nm in names(kdd_times))
  cat(sprintf('  %-20s %.2f sec\n', nm, kdd_times[[nm]]))


=== KDD Cup: glex timing ===
  PDP k=5              n_exp=5  n_bg=4898431  max_s=1 ... 519.56 sec
  PDP k=10             n_exp=10  n_bg=4898431  max_s=1 ... 519.03 sec
  PDP k=100            n_exp=100  n_bg=4898431  max_s=1 ... 519.20 sec
  Full PDP             n_exp=42  n_bg=4898431  max_s=1 ... 519.33 sec
  Joint PDP k=5        n_exp=175  n_bg=4898431  max_s=2 ... 568.10 sec

--- KDD Cup summary ---
  PDP k=5              519.56 sec
  PDP k=10             519.03 sec
  PDP k=100            519.20 sec
  Full PDP             519.33 sec
  Joint PDP k=5        568.10 sec


---
## Combined timing summary

In [16]:
experiments <- c('PDP k=5', 'PDP k=10', 'PDP k=100', 'Full PDP', 'Joint PDP k=5')

cat(sprintf('\n%-20s  %10s  %10s\n', 'Experiment', 'IEEE-CIS', 'KDD Cup'))
cat(strrep('-', 44), '\n')
for (exp in experiments) {
  cat(sprintf('%-20s  %9.2f s  %9.2f s\n',
              exp, ieee_times[[exp]], kdd_times[[exp]]))
}


Experiment              IEEE-CIS     KDD Cup
-------------------------------------------- 
PDP k=5                   98.56 s     519.56 s
PDP k=10                  98.82 s     519.03 s
PDP k=100                 99.09 s     519.20 s
Full PDP                  98.41 s     519.33 s
Joint PDP k=5            129.83 s     568.10 s
